# TARA Phase 3.4.1 — Gate 2: Colab / Google Drive Live Persistence Acceptance Test

This notebook verifies checkpoint selection isolation, genuine recovery from Drive fixtures,
real TaraStreamingJsonlDataset uncommitted iteration, ShardLedger persistence, and atomic pointer resolution across a live Colab restart.

### Acceptance Protocol
1. **Part 1 (Process A)**: Mount Google Drive, create synthetic fixtures on persistent Drive storage, train `stage_0` to step 10, train `stage_interrupted_test` to step 15, and crash runtime via `os._exit(0)`.
2. **Restart Runtime**: Abrupt runtime crash ensures zero memory/disk persistence in ephemeral `/content`.
3. **Part 2 (Process B)**: Fresh runtime restores `stage_0` and verifies step 10 (not step 15); verifies ShardLedger; restores `stage_interrupted_test` and verifies step 15; initializes `TaraStreamingJsonlDataset` from Drive fixtures and verifies remaining units (`line_10`..`line_13`) are trained without duplicating lines 0..9.
4. **Cleanup**: Cleans up test artifacts from Drive.

In [ ]:
# --- PART 1: Run in First Colab Session ---
from google.colab import drive
import os, sys, shutil, json

drive.mount('/content/drive')

GDRIVE_TEST_DIR = '/content/drive/MyDrive/tara_colab_persistence_test'
os.makedirs(GDRIVE_TEST_DIR, exist_ok=True)
print(f'[Colab Test] Persistent storage ready at: {GDRIVE_TEST_DIR}')

# Clone / cd to tara repository
%cd /content
if not os.path.exists('tara'):
    !git clone https://github.com/manjukp6-hue/tara.git
%cd /content/tara
!git checkout phase3-incremental-training
!git pull origin phase3-incremental-training
!pip install -q safetensors

# Requirement 9: Store synthetic test fixtures on Drive so fresh runtime has zero dependence on ephemeral /content
gdrive_fixtures_dir = os.path.join(GDRIVE_TEST_DIR, 'fixtures')
os.makedirs(gdrive_fixtures_dir, exist_ok=True)
shard_path = os.path.join(gdrive_fixtures_dir, 'shard_0.jsonl')
docs = [
    {'unit_id': f'u_{i}', 'text': f'TARA architectural durability validation doc unit {i}.'}
    for i in range(14)
]
with open(shard_path, 'w', encoding='utf-8') as f:
    for d in docs:
        f.write(json.dumps(d) + '\n')

tok_path = os.path.join(gdrive_fixtures_dir, 'tara_synthetic_tok.json')
vocab = {chr(i): i for i in range(32, 127)}
vocab['<|unk|>'] = 0
vocab['<|pad|>'] = 1
vocab['<|eos|>'] = 2
with open(tok_path, 'w', encoding='utf-8') as f:
    json.dump({'model': {'vocab': vocab}}, f)

out_dir_stage0 = os.path.join(GDRIVE_TEST_DIR, 'stage_0')
out_dir_interrupted = os.path.join(GDRIVE_TEST_DIR, 'stage_interrupted_test')
shutil.rmtree(out_dir_stage0, ignore_errors=True)
shutil.rmtree(out_dir_interrupted, ignore_errors=True)
os.makedirs(out_dir_stage0, exist_ok=True)
os.makedirs(out_dir_interrupted, exist_ok=True)

# Run Process A in isolated process
proc_a_script = f'''import os, sys, json, torch
sys.path.insert(0, '.')
from cloud_train import (
    TaraForCausalLM,
    TaraNativeTokenizer,
    save_checkpoint,
    ShardLedger,
)
tok_path = {repr(tok_path)}
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
vocab = tok_data['model']['vocab']
tok = TaraNativeTokenizer(vocab)
cfg = {{
    'vocab_size': 128, 'hidden_size': 64, 'intermediate_size': 128,
    'num_hidden_layers': 2, 'num_attention_heads': 2, 'num_key_value_heads': 2,
    'head_dim': 32, 'max_position_embeddings': 256, 'rms_norm_eps': 1e-5,
    'rope_theta': 10000.0, 'tie_word_embeddings': False,
}}
# Checkpoint A (stage_0): 10 steps
model_a = TaraForCausalLM(cfg)
opt_a = torch.optim.AdamW(model_a.parameters(), lr=1e-4)
scaler_a = torch.amp.GradScaler('cuda', enabled=False)
out_dir_a = {repr(out_dir_stage0)}
ledger_a = ShardLedger(out_dir_a)
applied_units = [f'u_{i}' for i in range(10)]
for u in applied_units:
    opt_a.zero_grad()
    dummy_x = torch.randint(0, 128, (1, 16))
    out = model_a(dummy_x)
    logits = out[0] if isinstance(out, tuple) else out
    loss = logits.sum() * 0.0 + 1.25
    loss.backward()
    opt_a.step()
cursor_a = {{
    'active_shard': 'shard_0.jsonl',
    'active_shard_line_offset': 10,
    'active_shard_byte_offset': 0,
    'active_chunk_idx': 0,
    'total_samples_trained': 10,
    'samples_consumed': 10,
    'completed_shards': [],
    'applied_unit_ids': applied_units,
}}
save_checkpoint(
    model_a, opt_a, scaler_a, cfg, tok_path, out_dir_a,
    step=10, epoch=0, loss=1.25, best_loss=1.25,
    stage_id='stage_0', data_cursor=cursor_a,
    shard_ledger=ledger_a, sync_to_gdrive=False,
)
# Checkpoint B (stage_interrupted_test): distinct step 15 and stage_id
model_b = TaraForCausalLM(cfg)
opt_b = torch.optim.AdamW(model_b.parameters(), lr=2e-4)
scaler_b = torch.amp.GradScaler('cuda', enabled=False)
out_dir_b = {repr(out_dir_interrupted)}
for _ in range(15):
    opt_b.zero_grad()
    dummy_x = torch.randint(0, 128, (1, 16))
    out = model_b(dummy_x)
    logits = out[0] if isinstance(out, tuple) else out
    loss = logits.sum() * 0.0 + 0.95
    loss.backward()
    opt_b.step()
cursor_b = {{
    'active_shard': 'shard_0.jsonl',
    'active_shard_line_offset': 15,
    'active_shard_byte_offset': 0,
    'active_chunk_idx': 0,
    'total_samples_trained': 15,
    'samples_consumed': 15,
    'completed_shards': [],
    'applied_unit_ids': [f'u_b_{i}' for i in range(15)],
}}
save_checkpoint(
    model_b, opt_b, scaler_b, cfg, tok_path, out_dir_b,
    step=15, epoch=1, loss=0.95, best_loss=0.95,
    stage_id='stage_interrupted_test', data_cursor=cursor_b,
    shard_ledger=None, sync_to_gdrive=False,
)
print('[Process A] Successfully committed stage_0 (step 10) and stage_interrupted_test (step 15) to Drive.')
'''
with open('/content/run_proc_a.py', 'w', encoding='utf-8') as f:
    f.write(proc_a_script)

print('[Colab Test Part 1] Launching Process A in isolated process...')
!python /content/run_proc_a.py

print('[Colab Test Part 1] Process A finished. Crashing runtime now to simulate preemption/restart...')
import os
os._exit(0)


### PART 2: Run in Fresh Colab Session (After Runtime Restart)

Execute the cell below in the newly spawned Colab runtime to verify recovery from Google Drive.

In [ ]:
# --- PART 2: Run in Fresh Colab Session (After Runtime Restart) ---
from google.colab import drive
import os, sys, shutil, json

drive.mount('/content/drive')
GDRIVE_TEST_DIR = '/content/drive/MyDrive/tara_colab_persistence_test'
out_dir_stage0 = os.path.join(GDRIVE_TEST_DIR, 'stage_0')
out_dir_interrupted = os.path.join(GDRIVE_TEST_DIR, 'stage_interrupted_test')
gdrive_fixtures_dir = os.path.join(GDRIVE_TEST_DIR, 'fixtures')

%cd /content
if not os.path.exists('tara'):
    !git clone https://github.com/manjukp6-hue/tara.git
%cd /content/tara
!git checkout phase3-incremental-training
!git pull origin phase3-incremental-training

import torch
from cloud_train import (
    TaraForCausalLM,
    TaraNativeTokenizer,
    TaraStreamingJsonlDataset,
    try_resume_from_checkpoint,
    ShardLedger,
)

print('[Process B] Starting post-crash recovery validation from Google Drive...')
tok_path = os.path.join(gdrive_fixtures_dir, 'tara_synthetic_tok.json')
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
tok = TaraNativeTokenizer(tok_data['model']['vocab'])

cfg = {
    'vocab_size': 128, 'hidden_size': 64, 'intermediate_size': 128,
    'num_hidden_layers': 2, 'num_attention_heads': 2, 'num_key_value_heads': 2,
    'head_dim': 32, 'max_position_embeddings': 256, 'rms_norm_eps': 1e-5,
    'rope_theta': 10000.0, 'tie_word_embeddings': False,
}

# 1. Verification of Checkpoint Selection Isolation & ShardLedger (stage_0 must restore step 10, NOT step 15)
model_stage0 = TaraForCausalLM(cfg)
opt_stage0 = torch.optim.AdamW(model_stage0.parameters(), lr=1e-4)
scaler_stage0 = torch.amp.GradScaler('cuda', enabled=False)
step0, _, best_loss0, was_resumed0, cursor0 = try_resume_from_checkpoint(
    out_dir_stage0, model_stage0, opt_stage0, scaler_stage0,
    tokenizer_path=tok_path,
    tokenizer=tok,
    expected_stage_id='stage_0',
)
print(f'[Process B: stage_0] Resumed: {was_resumed0}, Step: {step0}, Loss: {best_loss0}')
assert was_resumed0, 'stage_0 checkpoint was not resumed!'
assert step0 == 10, f'Checkpoint selection isolation failure: expected step 10 for stage_0, got {step0}'
assert cursor0['active_shard_line_offset'] == 10, f'Cursor offset mismatch: {cursor0}'
committed_units0 = cursor0.get('applied_unit_ids', [])
expected_units = [f'u_{i}' for i in range(10)]
assert committed_units0 == expected_units, f'Unexpected committed units: {committed_units0}'

# ShardLedger verification
ledger0 = ShardLedger(out_dir_stage0)
assert ledger0.last_committed_step == 10, f'Ledger step mismatch: {ledger0.last_committed_step}'
assert ledger0.active_shard_line_offset == 10, f'Ledger offset mismatch: {ledger0.active_shard_line_offset}'
assert ledger0.total_records_processed == 10, f'Ledger records mismatch: {ledger0.total_records_processed}'
print(f'[Process B: ShardLedger] Verified: last_committed_step=10, active_shard_line_offset=10, records=10')

# 2. Verification of stage_interrupted_test (must restore step 15 independently)
model_interrupted = TaraForCausalLM(cfg)
opt_interrupted = torch.optim.AdamW(model_interrupted.parameters(), lr=2e-4)
scaler_interrupted = torch.amp.GradScaler('cuda', enabled=False)
step_i, _, best_loss_i, was_resumed_i, cursor_i = try_resume_from_checkpoint(
    out_dir_interrupted, model_interrupted, opt_interrupted, scaler_interrupted,
    tokenizer_path=tok_path,
    tokenizer=tok,
    expected_stage_id='stage_interrupted_test',
)
print(f'[Process B: stage_interrupted] Resumed: {was_resumed_i}, Step: {step_i}, Loss: {best_loss_i}')
assert was_resumed_i, 'stage_interrupted_test checkpoint was not resumed!'
assert step_i == 15, f'Expected step 15 for stage_interrupted_test, got {step_i}'

# 3. Requirement 8: Use actual TaraStreamingJsonlDataset iterator to yield remaining uncommitted units
shard_path = os.path.join(gdrive_fixtures_dir, 'shard_0.jsonl')
dataset = TaraStreamingJsonlDataset(
    file_paths=[shard_path],
    tokenizer=tok,
    max_seq_len=64,
    resume_shard='shard_0.jsonl',
    resume_line_offset=cursor0['active_shard_line_offset'],
    resume_byte_offset=cursor0.get('active_shard_byte_offset', 0),
    resume_chunk_idx=cursor0.get('active_chunk_idx', 0),
)
iter_units = []
for item in dataset:
    input_ids, labels, shard_name, line_idx, line_byte, chunk_idx = item
    iter_units.append(f'line_{line_idx}_chunk_{chunk_idx}')
    opt_stage0.zero_grad()
    out = model_stage0(input_ids.unsqueeze(0))
    logits = out[0] if isinstance(out, tuple) else out
    loss = logits.sum() * 0.0 + 1.10
    loss.backward()
    opt_stage0.step()
    step0 += 1

print(f'[Process B] Real dataset iterator yielded remaining units: {iter_units}')
# In our 14-doc synthetic shard (lines 0..13), lines 0..9 were committed; lines 10..13 yielded
expected_iter_units = ['line_10_chunk_0', 'line_11_chunk_0', 'line_12_chunk_0', 'line_13_chunk_0']
assert iter_units == expected_iter_units, f'Iterator yielded unexpected units: {iter_units}'
assert step0 == 14, f'Expected final step 14, got {step0}'
print('[Process B] Verified zero committed units duplicated (u_0..u_9 preserved), uncommitted units u_10..u_13 trained to step 14, and ShardLedger verified!')

shutil.rmtree(GDRIVE_TEST_DIR, ignore_errors=True)
print('\n' + '='*60)
print('ALL COLAB / GOOGLE DRIVE DURABILITY ACCEPTANCE GATES PASSED!')
print('='*60)
